2025.12.20 시제 논문


In [2]:
import pandas as pd
import numpy as np

# =========================================================
# Weighted-variance (Gries-style dispersion) for "-었-" attachment rate
# - value per (verb, genre): p_{i,g} = k_{i,g} / n_{i,g}
#   n_{i,g} = sum(ID) over EF (all EP_T True/False)
#   k_{i,g} = sum(ID) over EF & EP_T==True
# - weights: w_{i,g} = n_{i,g} / sum_g n_{i,g}
# - weighted mean: p̄_i = sum_g w_{i,g} p_{i,g}
# - weighted variance: Disp_i = sum_g w_{i,g} (p_{i,g} - p̄_i)^2
# =========================================================

# ✅ Use the uploaded file in this environment:
data_file = r"csv\통계용\V_VX_E_category-었-결합_2025-12-19_17-53.csv"

# ---------------------------
# User-tunable settings
# ---------------------------
VERB_COL   = "V_No"
GENRE_COL  = "category"
EN_COL     = "EN_label"
EN_VAL     = "EF"
EP_COL     = "EP_T"
COUNT_COL  = "ID"

MIN_TOTAL_VERB  = 50   # 최소 전체(verb) 기회수 sum(ID)
MIN_TOTAL_CELL  = 20   # 최소 (verb,genre) 기회수 sum(ID)

# ---------------------------
# Core function
# ---------------------------
def compute_weighted_variance_dispersion(
    df: pd.DataFrame,
    verb_col: str = VERB_COL,
    genre_col: str = GENRE_COL,
    en_col: str = EN_COL,
    en_val: str = EN_VAL,
    ep_col: str = EP_COL,
    count_col: str = COUNT_COL,
    min_total_verb: float = MIN_TOTAL_VERB,
    min_total_cell: float = MIN_TOTAL_CELL,
):
    """
    Returns
    -------
    cell : DataFrame
        rows: (verb, genre)
        cols: n_ig, k_ig, p_ig, w_ig, pbar_i, dev, dev2, wdev2
    verb : DataFrame
        rows: verb
        cols: n_i, k_i, pbar_i, Disp_i (weighted variance), SD_i, genres_used
    """

    # 1) EF 환경만 필터 (EP_T True/False는 유지)
    d = df[df[en_col] == en_val].copy()

    # 2) n_{i,g}: EF 내 전체 기회수 (sum ID)
    n_ig = (
        d.groupby([verb_col, genre_col], dropna=False)[count_col]
         .sum()
         .rename("n_ig")
         .reset_index()
    )

    # 3) k_{i,g}: EF & EP_T==True 성공수 (sum ID)
    k_ig = (
        d[d[ep_col] == True]
        .groupby([verb_col, genre_col], dropna=False)[count_col]
        .sum()
        .rename("k_ig")
        .reset_index()
    )

    # 4) merge, fill missing k with 0
    cell = n_ig.merge(k_ig, on=[verb_col, genre_col], how="left")
    cell["k_ig"] = cell["k_ig"].fillna(0.0)

    # 5) 최소 셀 필터 (너무 작은 셀은 비율이 튀니까 제외)
    cell = cell[cell["n_ig"] >= min_total_cell].copy()

    # 6) verb 전체 n_i 계산(필터 후 셀 합으로 계산) + 최소 verb 필터
    n_i = cell.groupby(verb_col, as_index=False)["n_ig"].sum().rename(columns={"n_ig": "n_i"})
    cell = cell.merge(n_i, on=verb_col, how="left")
    cell = cell[cell["n_i"] >= min_total_verb].copy()

    # 7) p_{i,g} = k/n, w_{i,g} = n / sum_g n
    cell["p_ig"] = cell["k_ig"] / cell["n_ig"]
    cell["w_ig"] = cell["n_ig"] / cell["n_i"]

    # 8) pbar_i = sum_g w*p
    pbar = (
        cell.assign(wp=lambda x: x["w_ig"] * x["p_ig"])
            .groupby(verb_col, as_index=False)["wp"].sum()
            .rename(columns={"wp": "pbar_i"})
    )
    cell = cell.merge(pbar, on=verb_col, how="left")

    # 9) Disp_i = sum_g w*(p - pbar)^2
    cell["dev"] = cell["p_ig"] - cell["pbar_i"]
    cell["dev2"] = cell["dev"] ** 2
    cell["wdev2"] = cell["w_ig"] * cell["dev2"]

    verb = (
        cell.groupby(verb_col, as_index=False)
            .agg(
                n_i=("n_ig", "sum"),
                k_i=("k_ig", "sum"),
                pbar_i=("pbar_i", "first"),
                Disp_i=("wdev2", "sum"),
                genres_used=(genre_col, "count"),
            )
    )
    verb["SD_i"] = np.sqrt(verb["Disp_i"])

    # 보기 좋게 정렬 (dispersion 큰 동사 먼저)
    verb = verb.sort_values(["Disp_i", "n_i"], ascending=[False, False]).reset_index(drop=True)

    return cell, verb


# ---------------------------
# Run
# ---------------------------
df = pd.read_csv(data_file)

cell_disp, verb_disp = compute_weighted_variance_dispersion(df)

# 결과 확인
print(verb_disp.head(20))

# 필요하면 저장
cell_disp.to_csv("cell_dispersion_details.csv", index=False, encoding="utf-8-sig")
verb_disp.to_csv("verb_dispersion_weighted_variance.csv", index=False, encoding="utf-8-sig")


    V_No  n_i    k_i    pbar_i    Disp_i  genres_used      SD_i
0   3949   62   22.0  0.354839  0.163217            2  0.404002
1   3889  112   77.0  0.687500  0.156539            2  0.395650
2   3508  285  159.0  0.557895  0.148810            2  0.385759
3   3249  424  182.0  0.429245  0.147113            7  0.383554
4   2147   58   29.0  0.500000  0.145433            2  0.381356
5   3915   96   47.0  0.489583  0.140687            2  0.375082
6   3429  112   61.0  0.544643  0.137744            3  0.371139
7   2178   73   51.0  0.698630  0.135229            2  0.367736
8   3457   87   33.0  0.379310  0.134455            2  0.366681
9   3811   89   26.0  0.292135  0.132116            3  0.363478
10  3594   60   25.0  0.416667  0.129700            2  0.360139
11  3353  210  144.0  0.685714  0.127642            3  0.357270
12  3661  124   61.0  0.491935  0.126271            2  0.355346
13  3393  128   44.0  0.343750  0.123244            3  0.351061
14  2176   77   49.0  0.636364  0.118694